# Routes and parameters

A route is a function with a decorator: the method and the path.

In [1]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

JOKES = [
    {"id": 1, "artist": "Nora Lebrun", "text": "I told my plants a joke. Nothing grew."},
    {"id": 2, "artist": "Max Pellerin", "text": "My GPS has an accent."},
    {"id": 3, "artist": "Nora Lebrun", "text": "My alarm clock and I are not speaking."},
]

app = FastAPI()
client = TestClient(app)


@app.get("/jokes")
def list_jokes() -> list[dict]:
    return JOKES


print(len(client.get("/jokes").json()))

3


## Path parameters: `/jokes/{joke_id}`

A name between braces in the path becomes a parameter of the function. Its
annotation is **checked and converted** by FastAPI: `"2"` in the URL arrives
as the integer `2`.

In [2]:
@app.get("/jokes/{joke_id}")
def read_joke(joke_id: int) -> dict:
    for joke in JOKES:
        if joke["id"] == joke_id:
            return joke
    raise HTTPException(status_code=404, detail=f"No joke with id {joke_id}.")


print(client.get("/jokes/2").json())

{'id': 2, 'artist': 'Max Pellerin', 'text': 'My GPS has an accent.'}


In [3]:
response = client.get("/jokes/99")
print(response.status_code, response.json())

404 {'detail': 'No joke with id 99.'}


In [4]:
response = client.get("/jokes/two")
print(response.status_code)
print(response.json()["detail"][0]["msg"])

422
Input should be a valid integer, unable to parse string as an integer


`raise HTTPException(...)` stops the function and answers with this status
code; `detail` is the message. A text where an `int` is expected is refused
with a `422` before the function is even called.

## Query parameters: `/search?artist=nora&limit=1`

A parameter that is not in the path comes from the query string. With a
default value it is optional; `str | None = None` means *maybe absent*.

In [5]:
@app.get("/search")
def search(artist: str | None = None, limit: int = 10) -> list[dict]:
    result = []
    for joke in JOKES:
        if artist is None or artist.lower() in joke["artist"].lower():
            result.append(joke)
    return result[:limit]


print(len(client.get("/search").json()))
print(len(client.get("/search", params={"artist": "nora"}).json()))
print(client.get("/search?artist=nora&limit=1").json())

3
2
[{'id': 1, 'artist': 'Nora Lebrun', 'text': 'I told my plants a joke. Nothing grew.'}]


`params={...}` builds the `?artist=nora` part: the two last calls are the
same kind of request.

## Routes are tried in order

In [6]:
@app.get("/jokes/latest")
def latest() -> dict:
    return JOKES[-1]


response = client.get("/jokes/latest")
print(response.status_code)

422


`/jokes/{joke_id}` was declared first and matches `/jokes/latest` too:
FastAPI calls `read_joke("latest")`, which is not an `int`. A fixed path
must be declared **before** the path with a parameter.

## The documentation is generated

FastAPI describes every route from the decorators and the annotations. With
a server running, `/docs` shows this description as a page where each route
can be tried.

In [7]:
for path, methods in app.openapi()["paths"].items():
    print(path, list(methods))

/jokes ['get']
/jokes/{joke_id} ['get']
/search ['get']
/jokes/latest ['get']
